# Structured Sentiment Classifier

Build a reliable sentiment classifier that returns structured, typed outputs every time.

How can we classify customer reviews as positive, negative, or neutral with confidence scores - reliably, every time?

Traditional approaches to sentiment analysis often involve training custom models or using fragile prompt engineering with raw LLM calls. With raw prompting, you might get JSON sometimes, plain text other times, or malformed outputs that crash your application downstream. The output format is unpredictable, and there's no built-in validation or retry logic.

Mellea solves this by treating LLM calls as generative programs with structured outputs. You define the exact shape of data you want using Python types (Pydantic models), and Mellea guarantees that's what you get back - with automatic validation and repair if needed.

In this recipe, we will use the [Mellea](https://mellea.ai) library to build a sentiment classifier that returns structured, typed outputs.

Mellea supports several different LLM runtimes, called `backends`, to access an LLM. In this recipe, we will use the [LiteLLM](https://docs.litellm.ai/docs/) [backend](https://github.com/generative-computing/mellea/blob/main/mellea/backends/litellm.py). We can use the LiteLLM backend with providers supported by LiteLLM such as IBM [watsonx.ai](https://www.ibm.com/products/watsonx-ai). In this recipe, you will use the IBM® [Granite](https://www.ibm.com/granite) model now available on watsonx.ai.

# Steps

## Step 1. Set up your environment

While you can choose from several tools, this recipe is best suited for a Jupyter Notebook. Jupyter Notebooks are widely used within data science to combine code with various data sources such as text, images and data visualizations. 

You can run this notebook in [Colab](https://colab.research.google.com/), or download it to your system and [run the notebook locally](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started_with_Jupyter_Locally/Getting_Started_with_Jupyter_Locally.md). 

To avoid Python package dependency conflicts, we recommend setting up a [virtual environment](https://docs.python.org/3/library/venv.html).

Note, this notebook is compatible with Python 3.12, the default in Colab at the time of publishing this recipe. To check your python version, you can run the `!python --version` command in a code cell.

## Step 2. Set up a watsonx.ai instance

See [Getting Started with IBM watsonx](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_WatsonX.ipynb) for information on getting ready to use watsonx.ai. 

You will need three credentials from the watsonx.ai set up to add to your environment: `WATSONX_URL`, `WATSONX_APIKEY`, and `WATSONX_PROJECT_ID`.

## Step 3. Install relevant libraries and set up credentials and the Mellea session

We'll need a few libraries for this recipe. We will be using Mellea and LiteLLM to use Granite on watsonx.ai.

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    "mellea[litellm]"
! echo "::endgroup::"

Now we will get the credentials to use watsonx.ai and load them into the environment where LiteLLM will expect them.

In [ ]:
from ibm_granite_community.notebook_utils import get_env_var

# Load credentials into environment variables
get_env_var("WATSONX_API_KEY")
get_env_var("WATSONX_PROJECT_ID")
get_env_var("WATSONX_API_URL")

Now we can create a MelleaSession using the LiteLLM backend connected to the Granite model on watsonx.ai.

In [ ]:
from mellea import start_session, MelleaSession

m: MelleaSession = start_session(
        backend_name="litellm",
        model_id="watsonx/ibm/granite-4-h-small",
    )

## Step 4. The problem with raw prompting

Let's first see what happens when we try to classify sentiment using raw prompting without structure. We'll ask the model to classify a customer review.

In [ ]:
# Raw prompting approach - fragile and unpredictable
review = "This product exceeded my expectations! The quality is outstanding and delivery was fast."

raw_result = m.instruct(
    "Classify the sentiment of this review as positive, negative, or neutral. "
    "Also provide a confidence score between 0 and 1.\n\n"
    f"Review: {review}"
)

print("Raw prompting result:")
print(raw_result)
print(f"\nType: {type(raw_result)}")

The output is just a string. We'd need to parse it manually, handle different formats the model might return, and deal with edge cases where the output doesn't match our expectations. This is fragile and error-prone.

## Step 5. Define a structured output format with Pydantic

Now let's define exactly what we want using a Pydantic model. This creates a contract: "I want a SentimentResult object with these exact fields and types."

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class SentimentResult(BaseModel):
    """Structured sentiment analysis result"""
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="The overall sentiment of the text"
    )
    confidence: float = Field(
        ge=0.0, 
        le=1.0,
        description="Confidence score between 0 and 1"
    )
    reasoning: str = Field(
        description="Brief explanation of why this sentiment was assigned"
    )

The Pydantic model defines:
- `sentiment`: Must be exactly one of three values (positive, negative, neutral)
- `confidence`: Must be a float between 0.0 and 1.0
- `reasoning`: A string explaining the classification

## Step 6. Use Mellea to get structured output

Now we'll use Mellea's `format` parameter to request structured output. Mellea will ensure the model returns data that matches our Pydantic schema.

In [ ]:
def classify_sentiment(m: MelleaSession, review: str) -> SentimentResult:
    """Classify sentiment with structured output"""
    result = m.instruct(
        "Analyze the sentiment of the following customer review: {{review}}",
        user_variables={"review": review},
        format=SentimentResult,
    )
    return result

# Test with a positive review
positive_review = "This product exceeded my expectations! The quality is outstanding and delivery was fast."
result = classify_sentiment(m, positive_review)

print("Structured result:")
print(f"Sentiment: {result.sentiment}")
print(f"Confidence: {result.confidence}")
print(f"Reasoning: {result.reasoning}")
print(f"\nType: {type(result)}")

Notice that:
1. The result is a typed `SentimentResult` object, not a string
2. We can access fields directly: `result.sentiment`, `result.confidence`
3. The confidence is guaranteed to be between 0.0 and 1.0
4. The sentiment is guaranteed to be one of the three allowed values

## Step 7. Test with different review types

Let's test our classifier with various types of reviews to see how it handles different sentiments.

In [ ]:
# Test cases
reviews = [
    "Terrible experience. The product broke after one day and customer service was unhelpful.",
    "It's okay, nothing special. Does what it's supposed to do.",
    "Absolutely love it! Best purchase I've made this year. Highly recommend!",
    "The product arrived damaged and the return process was complicated.",
]

print("Sentiment Classification Results:\n")
print("=" * 80)

for i, review in enumerate(reviews, 1):
    result = classify_sentiment(m, review)
    print(f"\nReview {i}: {review[:60]}...")
    print(f"Sentiment: {result.sentiment.upper()}")
    print(f"Confidence: {result.confidence:.2f}")
    print(f"Reasoning: {result.reasoning}")
    print("-" * 80)

Each result is guaranteed to have the correct structure. No parsing errors, no unexpected formats, no crashes downstream.

## Step 8. Add validation requirements

We can add requirements to ensure the model's reasoning is thorough and the confidence score is appropriate.

In [ ]:
from mellea.stdlib.requirement import req, simple_validate

def classify_sentiment_with_validation(
    m: MelleaSession, 
    review: str
) -> SentimentResult:
    """Classify sentiment with validation requirements"""
    result = m.instruct(
        "Analyze the sentiment of the following customer review: {{review}}",
        user_variables={"review": review},
        format=SentimentResult,
        requirements=[
            req("The reasoning must be at least 10 words long"),
            req(
                "Confidence should be high (>0.7) for clearly positive or negative reviews",
                validation_fn=simple_validate(
                    lambda x: x.confidence > 0.7 if x.sentiment != "neutral" else True
                )
            ),
        ],
    )
    return result

# Test with a clearly positive review
clear_positive = "Absolutely amazing! Best product ever! I'm so happy with this purchase!"
result = classify_sentiment_with_validation(m, clear_positive)

print("Validated result:")
print(f"Sentiment: {result.sentiment}")
print(f"Confidence: {result.confidence}")
print(f"Reasoning: {result.reasoning}")
print(f"Reasoning word count: {len(result.reasoning.split())}")

The requirements ensure:
1. The reasoning is substantive (at least 10 words)
2. For non-neutral sentiments, confidence is appropriately high
3. If validation fails, Mellea automatically retries with repair prompts

<div class="alert alert-block alert-info">
ℹ️ 
Requirements in Mellea act as post-conditions that validate the LLM output. If validation fails, Mellea can automatically retry with repair prompts to guide the model toward valid output. This is the core of the Instruct-Validate-Repair pattern.
</div>

## Step 9. Batch processing with structured outputs

Let's process multiple reviews efficiently and collect structured results.

In [ ]:
def batch_classify_sentiments(
    m: MelleaSession, 
    reviews: list[str]
) -> list[SentimentResult]:
    """Classify multiple reviews and return structured results"""
    results = []
    for review in reviews:
        result = classify_sentiment(m, review)
        results.append(result)
    return results

# Sample customer reviews
customer_reviews = [
    "Great product, works perfectly!",
    "Disappointed with the quality.",
    "Average, nothing to complain about.",
    "Exceeded expectations in every way!",
    "Waste of money, very poor quality.",
]

# Process all reviews
results = batch_classify_sentiments(m, customer_reviews)

# Analyze the results
print("Batch Processing Results:\n")
print(f"Total reviews processed: {len(results)}")
print(f"Positive: {sum(1 for r in results if r.sentiment == 'positive')}")
print(f"Negative: {sum(1 for r in results if r.sentiment == 'negative')}")
print(f"Neutral: {sum(1 for r in results if r.sentiment == 'neutral')}")
print(f"Average confidence: {sum(r.confidence for r in results) / len(results):.2f}")

# Show detailed results
print("\nDetailed Results:")
print("=" * 80)
for review, result in zip(customer_reviews, results):
    print(f"\nReview: {review}")
    print(f"→ {result.sentiment.upper()} (confidence: {result.confidence:.2f})")

Because every result is a typed Python object, we can easily aggregate, filter, and analyze the data using standard Python operations.

## Step 10. Comparison: Raw vs Structured

Let's directly compare the raw prompting approach with Mellea's structured output approach.

In [ ]:
test_review = "The product is decent but overpriced for what you get."

print("=" * 80)
print("COMPARISON: Raw Prompting vs Mellea Structured Output")
print("=" * 80)

# Raw approach
print("\n1. RAW PROMPTING:")
print("-" * 80)
raw_output = m.instruct(
    "Classify sentiment as positive/negative/neutral with confidence 0-1.\n"
    f"Review: {test_review}"
)
print(f"Output: {raw_output}")
print(f"Type: {type(raw_output)}")
print("Issues:")
print("  ✗ Untyped string output")
print("  ✗ Requires manual parsing")
print("  ✗ No validation")
print("  ✗ Format may vary between calls")
print("  ✗ Downstream code must handle edge cases")

# Structured approach
print("\n2. MELLEA STRUCTURED OUTPUT:")
print("-" * 80)
structured_output = classify_sentiment(m, test_review)
print(f"Output: {structured_output}")
print(f"Type: {type(structured_output)}")
print(f"Sentiment: {structured_output.sentiment}")
print(f"Confidence: {structured_output.confidence}")
print(f"Reasoning: {structured_output.reasoning}")
print("Benefits:")
print("  ✓ Typed Python object")
print("  ✓ No parsing needed")
print("  ✓ Automatic validation")
print("  ✓ Guaranteed format")
print("  ✓ Safe for downstream code")

print("\n" + "=" * 80)

## Step 11. Conclusion

We've built a reliable sentiment classifier using Mellea's structured output capabilities. Here's what we learned:

**Key Takeaways:**

1. **Structured Outputs**: Using Pydantic models with `format=` ensures typed, validated outputs
2. **Type Safety**: Results are Python objects, not strings - no parsing needed
3. **Validation**: Requirements can enforce business rules and data quality
4. **Reliability**: Automatic retry and repair when validation fails
5. **Production Ready**: Structured outputs make downstream code safe and predictable

**Why This Matters:**

- **Raw prompting** works 70% of the time, silently fails 30%
- **Mellea structured outputs** validate, retry, and always return typed objects
- Your application code can safely assume the data structure is correct
- No defensive parsing, no try-catch blocks for malformed JSON

**Next Steps:**

Explore other Mellea recipes to learn about:
- [Instruct-Validate-Repair](../InstructValidateRepair/InstructValidateRepair.ipynb) - Deep dive into validation patterns
- Context Management - Threading state through multi-step operations
- RAG with Validated Citations - Building reliable question-answering systems

The pattern you learned here - defining structure with Pydantic and using `format=` - is fundamental to all reliable generative programs in Mellea.